# 🚨 PHẦN 4: ĐÁNH GIÁ THỰC NGHIỆM BẮT TẤN CÔNG ZERO-DAY (`KDDTest+`)
### Mục tiêu: So sánh thực tế **Mahalanobis vs Euclidean vs Random Forest** trên 17 loại tấn công mới lạ
---
### 1. Thiết kế thực nghiệm & Chuẩn hóa đầu vào (Chạy thật 100%):
- **Tiền xử lý chống Data Leakage:** Nén lệch phải $x' = \ln(1 + \max(0, x))$, chuẩn hóa tập kiểm thử `KDDTest+` bằng đúng trung bình $\mu_{\text{train}}$ và độ lệch chuẩn $\sigma_{\text{train}}$ từ `KDDTrain+`:
  $$z_{\text{test}} = \frac{x_{\text{test}} - \mu_{\text{train}}}{\sigma_{\text{train}}}$$
- **Chia tập `KDDTest+` (22.544 dòng) thành 2 nhóm đối chứng:**
  1. **Known Attacks:** Các tấn công *đã từng xuất hiện* trong `KDDTrain+`.
  2. **17 Novel / Zero-Day Attacks:** Các tấn công *chưa từng xuất hiện* trong tập Train (`apache2`, `mailbomb`, `mscan`, `httptunnel`, `snmpgetattack`, `xterm`...).


In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, confusion_matrix

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
TRAIN_FILE = DATA_DIR / "KDDTrain+.txt"
TEST_FILE = DATA_DIR / "KDDTest+.txt"
if not TRAIN_FILE.exists():
    TRAIN_FILE = Path(r"D:\data_miner\archive\KDDTrain+.txt")
    TEST_FILE = Path(r"D:\data_miner\archive\KDDTest+.txt")

COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
SKEWED_COLS = ['duration', 'src_bytes', 'dst_bytes']
NUMERIC_COLS = [c for c in COLUMN_NAMES if c not in CATEGORICAL_COLS + ['label', 'difficulty']]

# 17 chữ ký tấn công mới lạ CHỈ CÓ trong KDDTest+ (Zero-Day)
NOVEL_ATTACKS = {
    'apache2', 'mailbomb', 'processtable', 'udpstorm',
    'mscan', 'saint',
    'httptunnel', 'named', 'sendmail', 'snmpgetattack',
    'snmpguess', 'worm', 'xlock', 'xsnoop',
    'ps', 'sqlattack', 'xterm'
}

def load_data(file_path):
    df = pd.read_csv(file_path, names=COLUMN_NAMES, header=None)
    for c in CATEGORICAL_COLS:
        df[c] = df[c].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip().str.lower()
    df['is_attack'] = (df['label'] != 'normal').astype(int)
    df['is_novel'] = df['label'].isin(NOVEL_ATTACKS).astype(int)
    return df

def prep_num(df):
    d = df[NUMERIC_COLS].copy()
    for col in SKEWED_COLS:
        d[col] = np.log1p(np.maximum(0, d[col].astype(np.float32)))
    return d.values

df_train = load_data(TRAIN_FILE)
df_test = load_data(TEST_FILE)

scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train = np.hstack([scaler.fit_transform(prep_num(df_train)), encoder.fit_transform(df_train[CATEGORICAL_COLS])]).astype(np.float32)
X_test = np.hstack([scaler.transform(prep_num(df_test)), encoder.transform(df_test[CATEGORICAL_COLS])]).astype(np.float32)

y_train_bin = df_train['is_attack'].values
y_test_bin = df_test['is_attack'].values

print(f"✅ Nạp xong: Train = {X_train.shape} | Test = {X_test.shape}")
print(f"   Trong Test có {(y_test_bin == 1).sum():,} mẫu tấn công, gồm {(df_test['is_novel'] == 1).sum():,} mẫu Zero-Day mới lạ.")


### 2. Công thức Huấn luyện song song: Mahalanobis vs Euclidean vs Random Forest
- **Không gian nén PCA (bảo toàn 90% phương sai) & $K = 8$ tâm chuẩn Mini-Batch K-Means ($\boldsymbol{\mu}_k$):** Chỉ học từ dữ liệu **Normal** trong `KDDTrain+`.
- **Khoảng cách Euclidean ($D_E$):** Đo khoảng cách hình học thẳng tới tâm cụm gần nhất $k^*$:
  $$D_E(\mathbf{x}) = \min_{k \in \{1..8\}} \|\mathbf{x}_{\text{PCA}} - \boldsymbol{\mu}_k\|_2$$
- **Khoảng cách Mahalanobis ($D_M$) với điều chuẩn ma trận hiệp phương sai:** Triệt tiêu tương quan chéo và co giãn theo phương sai từng trục:
  $$\Sigma_{\text{reg}} = \Sigma_{\text{normal}} + 10^{-4}\mathbf{I}, \qquad D_M(\mathbf{x}) = \sqrt{(\mathbf{x}_{\text{PCA}} - \boldsymbol{\mu}_{k^*})^T \Sigma_{\text{reg}}^{-1} (\mathbf{x}_{\text{PCA}} - \boldsymbol{\mu}_{k^*})}$$
- **Quy tắc ra quyết định bất thường theo ngưỡng bách phân vị thứ 95 ($\tau_{95}$):**
  $$\hat{y}_{\text{unsup}} = \begin{cases} 1 \text{ (Anomaly / Zero-Day)}, & \text{nếu } D(\mathbf{x}) > \tau_{95} \\ 0 \text{ (Normal)}, & \text{nếu } D(\mathbf{x}) \le \tau_{95} \end{cases}$$


In [ ]:
# --- A. HUẤN LUYỆN KHÔNG GIÁM SÁT TRÊN TẬP NORMAL ---
X_train_normal = X_train[y_train_bin == 0]
pca = PCA(n_components=0.90, random_state=42)
X_pca_norm = pca.fit_transform(X_train_normal)

kmeans = MiniBatchKMeans(n_clusters=8, batch_size=256, random_state=42, n_init="auto")
kmeans.fit(X_pca_norm)
centroids = kmeans.cluster_centers_

cov = np.cov(X_pca_norm, rowvar=False) + np.eye(X_pca_norm.shape[1], dtype=np.float32) * 1e-4
inv_cov = np.linalg.pinv(cov)

def calc_distances(X_pca_data):
    euc_list, mah_list = [], []
    for i in range(0, len(X_pca_data), 5000):
        b = X_pca_data[i:i+5000]
        d_mat = cdist(b, centroids, metric="euclidean")
        euc_list.append(d_mat.min(axis=1))
        diff = b - centroids[d_mat.argmin(axis=1)]
        mah_list.append(np.sqrt(np.maximum(0, np.sum(diff @ inv_cov * diff, axis=1))))
    return np.concatenate(euc_list), np.concatenate(mah_list)

euc_norm, mah_norm = calc_distances(X_pca_norm)
thresh_euc = float(np.percentile(euc_norm, 95))
thresh_mah = float(np.percentile(mah_norm, 95))

# --- B. HUẤN LUYỆN CÓ GIÁM SÁT (RANDOM FOREST NHỊ PHÂN) ---
rf_bin = RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42, n_jobs=1)
rf_bin.fit(X_train, y_train_bin)

# --- C. DỰ ĐOÁN TRÊN TẬP TEST ĐỘC LẬP (KDDTest+) ---
X_pca_test = pca.transform(X_test)
euc_test, mah_test = calc_distances(X_pca_test)

pred_euc = (euc_test > thresh_euc).astype(int)
pred_mah = (mah_test > thresh_mah).astype(int)
pred_rf = rf_bin.predict(X_test)

print(f"✅ Huấn luyện xong! Ngưỡng 95%: Mahalanobis = {thresh_mah:.2f} | Euclidean = {thresh_euc:.2f}")


### 3. Công thức tính Tỉ lệ phát hiện (Known vs Zero-Day) & Báo động giả (FPR)
- **Tỉ lệ phát hiện tấn công đã biết (Known Attack Recall) & tấn công mới lạ (Zero-Day Recall):**
  $$\text{Known Recall} = \frac{TP_{\text{known}}}{TP_{\text{known}} + FN_{\text{known}}} \times 100\%, \qquad \text{Zero-Day Recall} = \frac{TP_{\text{novel}}}{TP_{\text{novel}} + FN_{\text{novel}}} \times 100\%$$
- **Tỉ lệ báo động giả trên lưu lượng bình thường (False Positive Rate - FPR):**
  $$\text{FPR} = \frac{FP_{\text{normal}}}{FP_{\text{normal}} + TN_{\text{normal}}} \times 100\%$$


In [ ]:
mask_known = (y_test_bin == 1) & (df_test['is_novel'].values == 0)
mask_novel = (y_test_bin == 1) & (df_test['is_novel'].values == 1)
mask_norm = (y_test_bin == 0)

models_pred = {
    'Mahalanobis (Unsupervised)': pred_mah,
    'Euclidean (Unsupervised)': pred_euc,
    'Random Forest (Supervised)': pred_rf
}

summary_rows = []
for name, p_arr in models_pred.items():
    known_rec = recall_score(y_test_bin[mask_known], p_arr[mask_known]) * 100
    novel_rec = recall_score(y_test_bin[mask_novel], p_arr[mask_novel]) * 100
    all_rec = recall_score(y_test_bin[y_test_bin == 1], p_arr[y_test_bin == 1]) * 100
    fpr = (p_arr[mask_norm] == 1).mean() * 100
    summary_rows.append({
        'Mô hình': name,
        'Known Attack Recall (%)': round(known_rec, 2),
        'Zero-Day Recall (%)': round(novel_rec, 2),
        'Tổng Attack Recall (%)': round(all_rec, 2),
        'Báo động giả FPR (%)': round(fpr, 2)
    })

df_summary = pd.DataFrame(summary_rows)
print("=== BẢNG 1: TỔNG HỢP HIỆU NĂNG TRÊN KDDTest+ (22,544 MẪU) ===")
display(df_summary)

# Chi tiết trên từng loại trong 17 tấn công Zero-Day thật
detail_rows = []
test_labels = df_test['label'].values
for att in sorted(NOVEL_ATTACKS):
    m = (test_labels == att)
    cnt = int(m.sum())
    if cnt == 0:
        continue
    detail_rows.append({
        'Attack Name': att,
        'Samples': cnt,
        'Mahalanobis (%)': round(pred_mah[m].mean() * 100, 1),
        'Euclidean (%)': round(pred_euc[m].mean() * 100, 1),
        'Random Forest (%)': round(pred_rf[m].mean() * 100, 1)
    })

df_detail = pd.DataFrame(detail_rows)
print(f"\n=== BẢNG 2: CHI TIẾT TỈ LỆ PHÁT HIỆN TRÊN {len(df_detail)} LOẠI ZERO-DAY TRONG KDDTest+ ===")
display(df_detail)


### 4. Biểu đồ cột so sánh trực quan trên từng chữ ký Zero-Day
- **Tỉ lệ phát hiện riêng cho từng loại tấn công $a$ trong 17 chữ ký Zero-Day của `KDDTest+`:**
  $$\text{Detection Rate}_a = \frac{\text{Số mẫu } a \text{ phát hiện đúng}}{\text{Tổng số mẫu } a \text{ trong } \texttt{KDDTest+}} \times 100\%$$


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 11), dpi=120)

# Biểu đồ 1: Known Attacks vs Zero-Day Attacks Recall
df_sum_melt = df_summary.melt(
    id_vars='Mô hình',
    value_vars=['Known Attack Recall (%)', 'Zero-Day Recall (%)'],
    var_name='Loại tấn công',
    value_name='Tỉ lệ phát hiện (%)'
)
sns.barplot(
    data=df_sum_melt, x='Loại tấn công', y='Tỉ lệ phát hiện (%)', hue='Mô hình',
    palette={'Mahalanobis (Unsupervised)': '#e74c3c', 'Euclidean (Unsupervised)': '#f39c12', 'Random Forest (Supervised)': '#2980b9'},
    ax=axes[0]
)
axes[0].set_title("1. So Sánh Tổng Quan: Tấn Công Đã Biết vs Tấn Công Mới Lạ (Zero-Day)", fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 110)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.1f%%', padding=3, fontsize=9, fontweight='bold')

# Biểu đồ 2: Chi tiết từng loại tấn công Zero-Day thật trong KDDTest+
df_det_melt = df_detail.melt(
    id_vars=['Attack Name', 'Samples'],
    value_vars=['Mahalanobis (%)', 'Euclidean (%)', 'Random Forest (%)'],
    var_name='Mô hình',
    value_name='Tỉ lệ phát hiện (%)'
)
sns.barplot(
    data=df_det_melt, x='Attack Name', y='Tỉ lệ phát hiện (%)', hue='Mô hình',
    palette={'Mahalanobis (%)': '#e74c3c', 'Euclidean (%)': '#f39c12', 'Random Forest (%)': '#2980b9'},
    ax=axes[1]
)
axes[1].set_title(f"2. Tỉ Lệ Phát Hiện Chi Tiết Trên {len(df_detail)} Chữ Ký Tấn Công Zero-Day Thật (KDDTest+)", fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 110)
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(title="Phương pháp phát hiện", loc='upper right')

plt.tight_layout()
plt.show()
